Copyright **`(c)`** 2025 Giovanni Squillero `<giovanni.squillero@polito.it>`  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free under certain conditions — see the [`license`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE.md) for details.  

In [6]:
from itertools import product, combinations
import numpy as np
import networkx as nx
from tqdm.auto import tqdm
from icecream import ic

/Users/hassine_elghazel/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def create_problem(
    size,
    *,
    density=1.0,
    negative_values=False,
    noise_level=0.0,
    seed=42,
):
    rng = np.random.default_rng(seed)
    map_data = rng.random(size=(size, 2))
    problem = rng.random((size, size))
    if negative_values:
        problem = problem * 2 - 1
    problem *= noise_level
    for a in range(size):
        for b in range(size):
            if rng.random() < density:
                problem[a, b] += np.sqrt(
                    np.square(map_data[a, 0] - map_data[b, 0]) +
                    np.square(map_data[a, 1] - map_data[b, 1])
                )
            else:
                problem[a, b] = np.inf
    np.fill_diagonal(problem, 0)
    return (problem * 1_000).round()


In [8]:
def build_graph(problem):
    masked = np.ma.masked_array(problem, mask=np.isinf(problem))
    G = nx.from_numpy_array(masked, create_using=nx.DiGraph)
    return G


In [9]:
def analyze_instance(size, density, noise_level, negative_values, seed=42, print_examples=False):
    problem = create_problem(
        size=size,
        density=density,
        negative_values=negative_values,
        noise_level=noise_level,
        seed=seed,
    )
    G = build_graph(problem)
    n = problem.shape[0]
    positive_count = 0
    best_cost = np.inf
    best_pair = (None, None)

    for s in range(n):
        try:
            if negative_values:
                dist = nx.single_source_bellman_ford_path_length(G, s, weight="weight")
            else:
                dist = nx.single_source_dijkstra_path_length(G, s, weight="weight")
        except nx.NetworkXUnbounded:
            continue

        for d, cost in dist.items():
            if s == d:
                continue
            if s < d and np.isfinite(cost) and cost > 0:
                positive_count += 1
                if print_examples:
                    ic(size, density, noise_level, negative_values, s, d, cost)
            if np.isfinite(cost) and cost < best_cost:
                best_cost = cost
                best_pair = (s, d)

    best_path = None
    if best_pair[0] is not None:
        s_best, d_best = best_pair
        try:
            if negative_values:
                best_path = nx.bellman_ford_path(G, s_best, d_best, weight="weight")
            else:
                best_path = nx.dijkstra_path(G, s_best, d_best, weight="weight")
        except (nx.NetworkXNoPath, nx.NetworkXUnbounded):
            best_path = None

    best_info = {
        "s": best_pair[0],
        "d": best_pair[1],
        "path": best_path,
        "cost": best_cost,
    }

    return problem, G, positive_count, best_info

In [ ]:
problem_10, G_10, positive_10, best_10 = analyze_instance(
    size=10,
    density=0.15,
    noise_level=10.0,
    negative_values=True,
    seed=42,
    print_examples=True,
)

ic("example positive_count", positive_10)
ic("example best", best_10)

sizes = [10, 20, 50, 100, 200, 500, 1000]
densities = [0.2, 0.5, 0.8, 1.0]
noise_levels = [0.0, 0.1, 0.5, 0.8]
negative_flags = [False, True]

results_summary = []
configs = list(product(sizes, densities, noise_levels, negative_flags))

for size, density, noise_level, neg in tqdm(
    configs,
    total=len(configs),
    desc="Configs",
):
    seed = hash((size, density, noise_level, neg)) % (2**32)
    print(f"\n=== size={size}, density={density}, noise={noise_level}, negative={neg} ===")
    problem, G, positive_count, best_info = analyze_instance(
        size=size,
        density=density,
        noise_level=noise_level,
        negative_values=neg,
        seed=seed,
        print_examples=False,
    )
    results_summary.append(
        (size, density, noise_level, neg, positive_count, best_info["cost"])
    )
    print(f"Number of pairs with a positive shortest path: {positive_count}")
    print(f"Best cost: {best_info['cost']} from {best_info['s']} to {best_info['d']}")

ic| 'example positive_count', positive_10: 0
ic| "example best": 'example best'
    best_10: {'cost': inf, 'd': None, 'path': None, 's': None}
Configs:  26%|██▌       | 58/224 [00:00<00:00, 560.48it/s]


=== size=10, density=0.2, noise=0.0, negative=False ===
Number of pairs with a positive shortest path: 15
Best cost: 228.0 from 3 to 2

=== size=10, density=0.2, noise=0.0, negative=True ===
Number of pairs with a positive shortest path: 36
Best cost: 94.0 from 6 to 3

=== size=10, density=0.2, noise=0.1, negative=False ===
Number of pairs with a positive shortest path: 7
Best cost: 154.0 from 3 to 6

=== size=10, density=0.2, noise=0.1, negative=True ===
Number of pairs with a positive shortest path: 29
Best cost: 212.0 from 0 to 7

=== size=10, density=0.2, noise=0.5, negative=False ===
Number of pairs with a positive shortest path: 42
Best cost: 360.0 from 0 to 2

=== size=10, density=0.2, noise=0.5, negative=True ===
Number of pairs with a positive shortest path: 16
Best cost: -467.0 from 3 to 8

=== size=10, density=0.2, noise=0.8, negative=False ===
Number of pairs with a positive shortest path: 18
Best cost: 137.0 from 3 to 5

=== size=10, density=0.2, noise=0.8, negative=True 

Configs:  51%|█████▏    | 115/224 [00:03<00:04, 25.99it/s]

Number of pairs with a positive shortest path: 4950
Best cost: 18.0 from 30 to 34

=== size=100, density=0.8, noise=0.1, negative=True ===
Number of pairs with a positive shortest path: 0
Best cost: inf from None to None

=== size=100, density=0.8, noise=0.5, negative=False ===
Number of pairs with a positive shortest path: 4950
Best cost: 24.0 from 89 to 54

=== size=100, density=0.8, noise=0.5, negative=True ===
Number of pairs with a positive shortest path: 0
Best cost: inf from None to None

=== size=100, density=0.8, noise=0.8, negative=False ===
Number of pairs with a positive shortest path: 4950
Best cost: 48.0 from 34 to 30

=== size=100, density=0.8, noise=0.8, negative=True ===
Number of pairs with a positive shortest path: 0
Best cost: inf from None to None

=== size=100, density=1.0, noise=0.0, negative=False ===
Number of pairs with a positive shortest path: 4950
Best cost: 5.0 from 18 to 61

=== size=100, density=1.0, noise=0.0, negative=True ===
Number of pairs with a po

Configs:  62%|██████▏   | 139/224 [00:14<00:12,  7.06it/s]

...

Number of pairs with a positive shortest path: 499500
Best cost: 14.0 from 319 to 281

=== size=1000, density=0.2, noise=0.8, negative=True ===


Configs:  89%|████████▉ | 200/224 [15:18<12:51, 32.16s/it]

Number of pairs with a positive shortest path: 0
Best cost: inf from None to None

=== size=1000, density=0.5, noise=0.0, negative=False ===


Configs:  90%|████████▉ | 201/224 [17:50<26:03, 67.99s/it]

Number of pairs with a positive shortest path: 499500
Best cost: 1.0 from 56 to 549

=== size=1000, density=0.5, noise=0.0, negative=True ===
